In [1]:
# Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score,
                             accuracy_score, f1_score, roc_auc_score)


print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Load Datasets (Math + Portuguese)
try:
    df_mat = pd.read_csv("student-mat.csv", sep=';')
    df_por = pd.read_csv("student-por.csv", sep=';')

    # Combine both datasets
    df = pd.concat([df_mat, df_por], ignore_index=True)

    print("Datasets loaded successfully!")
    print(f"Math dataset shape       : {df_mat.shape}")
    print(f"Portuguese dataset shape : {df_por.shape}")
    print(f"Combined dataset shape   : {df.shape}")

except FileNotFoundError:
    print("ERROR: Please place both 'student-mat.csv' and 'student-por.csv' in the same folder!")

Datasets loaded successfully!
Math dataset shape       : (395, 33)
Portuguese dataset shape : (649, 33)
Combined dataset shape   : (1044, 33)


In [3]:
# Select Only Important Features + Preprocessing

# We use only the most important features for better understanding and practical input
important_features = [
    'age', 'Medu', 'Fedu', 'studytime', 'failures',
    'absences', 'goout', 'Dalc', 'Walc', 'health', 'G1', 'G2'
]

# Encode categorical columns
for col in df.select_dtypes(include='object').columns:
    df[col] = LabelEncoder().fit_transform(df[col])

# Create Pass/Fail target (G3 >= 10 → Pass)
df['pass'] = (df['G3'] >= 10).astype(int)

# Final feature matrix and targets
X = df[important_features]
y_reg = df['G3']          # For Regression (exact score)
y_clf = df['pass']        # For Classification (Pass/Fail)

print("Important features selected:")
print(important_features)
print(f"\nFeature matrix shape : {X.shape}")

Important features selected:
['age', 'Medu', 'Fedu', 'studytime', 'failures', 'absences', 'goout', 'Dalc', 'Walc', 'health', 'G1', 'G2']

Feature matrix shape : (1044, 12)


In [4]:
# Split Dataset into Training and Testing sets

# Split for Regression
X_train, X_test, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42)

# Split for Classification
X_train_c, X_test_c, y_train_clf, y_test_clf = train_test_split(
    X, y_clf, test_size=0.2, random_state=42)

# Feature Scaling (needed for Linear & Logistic Regression)
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
X_train_cs = scaler.transform(X_train_c)
X_test_cs  = scaler.transform(X_test_c)

print("Dataset split completed!")
print(f"Training samples : {X_train.shape[0]}")
print(f"Testing samples  : {X_test.shape[0]}")

Dataset split completed!
Training samples : 835
Testing samples  : 209


In [5]:
#  Train All 4 Algorithms Individually

# ----- REGRESSION MODELS -----
lr = LinearRegression()
lr.fit(X_train_s, y_train_reg)

rf_reg = RandomForestRegressor(n_estimators=150, max_depth=12, random_state=42)
rf_reg.fit(X_train, y_train_reg)

# ----- CLASSIFICATION MODELS -----
log_reg = LogisticRegression(max_iter=1000, random_state=42)
log_reg.fit(X_train_cs, y_train_clf)

rf_clf = RandomForestClassifier(n_estimators=150, max_depth=12, random_state=42)
rf_clf.fit(X_train_c, y_train_clf)

print("All 4 algorithms trained successfully!")
print("1. Linear Regression")
print("2. Random Forest Regressor")
print("3. Logistic Regression")
print("4. Random Forest Classifier")

All 4 algorithms trained successfully!
1. Linear Regression
2. Random Forest Regressor
3. Logistic Regression
4. Random Forest Classifier


In [6]:
# Evaluate Models and Decide Best Algorithms

def get_reg_metrics(model, X, y):
    pred = model.predict(X)
    return {
        'MAE': mean_absolute_error(y, pred),
        'RMSE': np.sqrt(mean_squared_error(y, pred)),
        'R2': r2_score(y, pred)
    }

def get_clf_metrics(model, X, y):
    pred = model.predict(X)
    proba = model.predict_proba(X)[:, 1]
    return {
        'Accuracy': accuracy_score(y, pred),
        'F1': f1_score(y, pred),
        'ROC-AUC': roc_auc_score(y, proba)
    }

# Calculate metrics
reg_results = {
    'Linear Regression': get_reg_metrics(lr, X_test_s, y_test_reg),
    'Random Forest Regressor': get_reg_metrics(rf_reg, X_test, y_test_reg)
}

clf_results = {
    'Logistic Regression': get_clf_metrics(log_reg, X_test_cs, y_test_clf),
    'Random Forest Classifier': get_clf_metrics(rf_clf, X_test_c, y_test_clf)
}

# Decide Best Algorithms
best_reg = max(reg_results, key=lambda k: reg_results[k]['R2'])
best_clf = max(clf_results, key=lambda k: clf_results[k]['ROC-AUC'])

print("=" * 70)
print("MODEL PERFORMANCE SUMMARY")
print("=" * 70)

print("\nREGRESSION RESULTS:")
for name, m in reg_results.items():
    mark = "  ← BEST" if name == best_reg else ""
    print(f"  {name:28s}  R² = {m['R2']:.3f}   MAE = {m['MAE']:.3f}{mark}")

print("\nCLASSIFICATION RESULTS:")
for name, m in clf_results.items():
    mark = "  ← BEST" if name == best_clf else ""
    print(f"  {name:28s}  Accuracy = {m['Accuracy']:.3f}   ROC-AUC = {m['ROC-AUC']:.3f}{mark}")

print(f"\nBest Regression Algorithm     → {best_reg}")
print(f"Best Classification Algorithm → {best_clf}")

MODEL PERFORMANCE SUMMARY

REGRESSION RESULTS:
  Linear Regression             R² = 0.801   MAE = 0.978
  Random Forest Regressor       R² = 0.801   MAE = 0.965  ← BEST

CLASSIFICATION RESULTS:
  Logistic Regression           Accuracy = 0.871   ROC-AUC = 0.963  ← BEST
  Random Forest Classifier      Accuracy = 0.871   ROC-AUC = 0.952

Best Regression Algorithm     → Random Forest Regressor
Best Classification Algorithm → Logistic Regression


In [7]:
#  Function to Collect Student Details (User Input)

def get_value(prompt, default, min_val=None, max_val=None):
    while True:
        raw = input(f"{prompt} [{default}]: ").strip()
        if raw == "":
            return default
        try:
            val = int(raw)
            if min_val is not None and val < min_val:
                print(f"  → Value should be >= {min_val}")
                continue
            if max_val is not None and val > max_val:
                print(f"  → Value should be <= {max_val}")
                continue
            return val
        except ValueError:
            print("  → Please enter a valid integer.")

def collect_student():
    print("\n" + "=" * 70)
    print("ENTER STUDENT DETAILS ")
    print("Press ENTER to keep the default value shown in [brackets]")
    print("=" * 70)

    student = {}
    student['age']       = get_value("Age (15-22)", 17, 15, 22)
    student['Medu']      = get_value("Mother education (0=none ... 4=higher)", 3, 0, 4)
    student['Fedu']      = get_value("Father education (0=none ... 4=higher)", 3, 0, 4)
    student['studytime'] = get_value("Weekly study time (1=<2h, 2=2-5h, 3=5-10h, 4=>10h)", 2, 1, 4)
    student['failures']  = get_value("Number of past class failures (0-3)", 0, 0, 3)
    student['absences']  = get_value("Number of school absences (0-93)", 4, 0, 93)
    student['goout']     = get_value("Going out with friends (1=very low ... 5=very high)", 3, 1, 5)
    student['Dalc']      = get_value("Workday alcohol consumption (1=very low ... 5=very high)", 1, 1, 5)
    student['Walc']      = get_value("Weekend alcohol consumption (1=very low ... 5=very high)", 1, 1, 5)
    student['health']    = get_value("Current health status (1=very bad ... 5=very good)", 4, 1, 5)
    student['G1']        = get_value("First period grade G1 (0-20)", 12, 0, 20)
    student['G2']        = get_value("Second period grade G2 (0-20)", 13, 0, 20)

    return student

print("Input function ready!")

Input function ready!


In [8]:
#  Prediction Function

def predict(student, choice):
    new_df = pd.DataFrame([student])
    new_X = new_df[important_features]
    new_X_s = scaler.transform(new_X)

    print("\n" + "-" * 60)

    if choice == 1:
        pred = lr.predict(new_X_s)[0]
        print("Selected Algorithm : Linear Regression")
        print(f"Predicted Final Grade (G3) : {pred:.2f} / 20")
        print(f"Result : {'PASS' if pred >= 10 else 'FAIL'}")

    elif choice == 2:
        pred = rf_reg.predict(new_X)[0]
        print("Selected Algorithm : Random Forest Regressor")
        print("Parameters used    : n_estimators=150, max_depth=12")
        print(f"Predicted Final Grade (G3) : {pred:.2f} / 20")
        print(f"Result : {'PASS' if pred >= 10 else 'FAIL'}")

    elif choice == 3:
        pred = log_reg.predict(new_X_s)[0]
        proba = log_reg.predict_proba(new_X_s)[0][1]
        print("Selected Algorithm : Logistic Regression")
        print(f"Predicted           : {'PASS' if pred == 1 else 'FAIL'}")
        print(f"Probability of Pass : {proba:.1%}")

    elif choice == 4:
        pred = rf_clf.predict(new_X)[0]
        proba = rf_clf.predict_proba(new_X)[0][1]
        print("Selected Algorithm : Random Forest Classifier")
        print("Parameters used    : n_estimators=150, max_depth=12")
        print(f"Predicted           : {'PASS' if pred == 1 else 'FAIL'}")
        print(f"Probability of Pass : {proba:.1%}")

    print("-" * 60)

print("Prediction function ready!")

Prediction function ready!


In [9]:
# Main Interactive Prediction Flow

print("\n" + "=" * 70)
print("STARTING INTERACTIVE PREDICTION")
print("=" * 70)

# Take student details once
student =collect_student()

used_algorithms = set()

while True:
    print("\n" + "=" * 70)
    print("SELECT ALGORITHM TO PREDICT")
    print("=" * 70)
    print("  1 → Linear Regression          (Regression - exact score)")
    print("  2 → Random Forest Regressor    (Regression - exact score)")
    print("  3 → Logistic Regression        (Classification - Pass/Fail)")
    print("  4 → Random Forest Classifier   (Classification - Pass/Fail)")
    print("  0 → Finish and show Best Algorithms")
    print("-" * 70)

    choice = input("Enter your choice (0-4): ").strip()

    if choice == '0':
        break

    if choice not in ['1', '2', '3', '4']:
        print("Invalid choice! Please enter 0-4.")
        continue

    if choice in used_algorithms:
        print("You already used this algorithm. Please choose another one.")
        continue

    # Perform prediction
    predict(student, int(choice))
    used_algorithms.add(choice)

    if len(used_algorithms) == 4:
        print("\nAll 4 algorithms have been used.")
        break

    cont = input("\nDo you want to try another algorithm? (y/n): ").strip().lower()
    if cont != 'y':
        break


STARTING INTERACTIVE PREDICTION

ENTER STUDENT DETAILS 
Press ENTER to keep the default value shown in [brackets]
Age (15-22) [17]: 16
Mother education (0=none ... 4=higher) [3]: 0
Father education (0=none ... 4=higher) [3]: 0
Weekly study time (1=<2h, 2=2-5h, 3=5-10h, 4=>10h) [2]: 0
  → Value should be >= 1
Weekly study time (1=<2h, 2=2-5h, 3=5-10h, 4=>10h) [2]: 1
Number of past class failures (0-3) [0]: 2
Number of school absences (0-93) [4]: 25
Going out with friends (1=very low ... 5=very high) [3]: 2
Workday alcohol consumption (1=very low ... 5=very high) [1]: 2
Weekend alcohol consumption (1=very low ... 5=very high) [1]: 1
Current health status (1=very bad ... 5=very good) [4]: 1
First period grade G1 (0-20) [12]: 2
Second period grade G2 (0-20) [13]: 2

SELECT ALGORITHM TO PREDICT
  1 → Linear Regression          (Regression - exact score)
  2 → Random Forest Regressor    (Regression - exact score)
  3 → Logistic Regression        (Classification - Pass/Fail)
  4 → Random For

In [10]:
#  Final Best Algorithm Summary

print("\n" + "=" * 70)
print("FINAL RESULT - BEST ALGORITHMS")
print("=" * 70)

print(f"\nBest REGRESSION Algorithm     : {best_reg}")
print(f"  Decision Factor (R² Score)  : {reg_results[best_reg]['R2']:.3f}")
print(f"  Mean Absolute Error (MAE)   : {reg_results[best_reg]['MAE']:.3f}")
print("  Why chosen?")
print("  → Highest R² means this model best explains the variation in final grade (G3).")

print(f"\nBest CLASSIFICATION Algorithm : {best_clf}")
print(f"  Decision Factor (ROC-AUC)   : {clf_results[best_clf]['ROC-AUC']:.3f}")
print(f"  Accuracy                    : {clf_results[best_clf]['Accuracy']:.3f}")
print("  Why chosen?")
print("  → Highest ROC-AUC means this model best distinguishes between Pass and Fail students.")

print("\n" + "=" * 70)
print("Case Study Completed Successfully!")
print("=" * 70)


FINAL RESULT - BEST ALGORITHMS

Best REGRESSION Algorithm     : Random Forest Regressor
  Decision Factor (R² Score)  : 0.801
  Mean Absolute Error (MAE)   : 0.965
  Why chosen?
  → Highest R² means this model best explains the variation in final grade (G3).

Best CLASSIFICATION Algorithm : Logistic Regression
  Decision Factor (ROC-AUC)   : 0.963
  Accuracy                    : 0.871
  Why chosen?
  → Highest ROC-AUC means this model best distinguishes between Pass and Fail students.

Case Study Completed Successfully!
